# 02 — Rank test videos and save the submission files

Loads the **trained PCA + Ridge** model from notebook 01 and scores its predicted video-evoked EEG against the supplied `tst/sub-XX_tst.npy` recordings.

**Critical fix:** unlike the original notebook, no `argsort(gt)` is ever applied to test EEG. Every ranked row `i` refers to row `i` in the original `tst` array. Only subjects 01–04 have validation labels; subject 05 and 06 ground truth is never read.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
from frame_baseline import (
    TEST_IDS, extract_video_features, load_test_eeg,
    similarity, ranked_video_ids, validate_rankings, assert_submission,
)

ROOT = Path('.').resolve()
MODELS = ROOT / 'models'

# Cached features created by notebook 01; they can also be regenerated here.
test_frames = extract_video_features(
    ROOT / 'stimuli_test', TEST_IDS, ROOT / 'cache' / 'alexnet_test_frames.npz'
)
print('Candidate videos:', len(TEST_IDS), 'frame feature shape:', test_frames.shape)

## 1. Check the original-order prediction code on development subjects

These scores are validation results, not training accuracy. They do not change any model weights.

In [ ]:
for sub in range(1, 5):
    encoder = joblib.load(MODELS / f'encoder_sub-{sub:02d}.joblib')
    tst_eeg = load_test_eeg(ROOT, sub)         # NEVER sort test EEG by labels
    scores = similarity(encoder, tst_eeg, test_frames)
    ranked = ranked_video_ids(scores, TEST_IDS)
    assert_submission(ranked)

    truth = np.load(ROOT / 'tst_stim' / f'sub-{sub:02d}_tst_stim.npy')
    metrics = validate_rankings(ranked, truth)  # truth is already in original row order
    print(f"sub-{sub:02d} top1={metrics['top1']:.3f} "
          f"top5={metrics['top5']:.3f} "
          f"median-rank={metrics['median_rank']:.1f}")

## 2. Create exactly two competition files

Shape **(102,102)**, integer video IDs **1001–1102**, each ID exactly once per row, and rows in **original `tst` recording order**. No global matching/Hungarian assignment.

In [ ]:
for sub in (5, 6):
    encoder = joblib.load(MODELS / f'encoder_sub-{sub:02d}.joblib')
    tst_eeg = load_test_eeg(ROOT, sub)
    scores = similarity(encoder, tst_eeg, test_frames)
    ranked = ranked_video_ids(scores, TEST_IDS).astype(np.int32)
    assert_submission(ranked)

    output_file = ROOT / f'sub-{sub:02d}_ranked_candidate_list.npy'
    np.save(output_file, ranked)
    reloaded = np.load(output_file)
    assert_submission(reloaded)
    print(f'Saved {output_file.name} | shape={reloaded.shape} | '
          f'dtype={reloaded.dtype} | first three IDs in row 0={reloaded[0, :3]}')

print('Complete: submit the two .npy files printed above.')